# Building out Makemore MLP
### This time around, we give in input 3 characters, and expect a fourth char to be generated

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# now, we need to construct the dataset.
block_size = 3
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


In [5]:
# building out train, test, and val datasets.

X, Y = build_dataset(words)

torch.Size([228146, 3]) torch.Size([228146])


In [6]:
# now, we have to embed each character as an x-dimensional tensor.
# what even is an embedding? it's a random vector that we initialize for each character. initially, it contains nonsense values.
# but, as we train the model, the embeddings will be updated, and they will start to actually represent the chars meaningfully.
# why are we doing this instead of just one-hot encoding the chars?
# because, when we have chars in latent space embedded close to each other, we know that they are somewhat semantically related (in some words atleast), and they can be the catalyst for the model to generate unique and new results.

# ok, so, how are we going to implement this?
# we will have C, an embedding tensor that contains the embeddings of all 3 characters being given in input.

C = torch.randn((27, 2), requires_grad=True)
W1 = torch.randn(6, 100, requires_grad=True) # hidden layer weights.
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn(100, 27, requires_grad=True)
b2 = torch.randn(27, requires_grad=True)

params = [C, W1, b1, W2, b2]

In [8]:
# so, what is C[X] actually doing?
# so, it takes C, which is a (27, 2) tensor, and X, 

emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [ ]:
# now, each chars embedding is supposed to be (1, 3). And, each tri-gram's embedding is a concatenation of it's constituent 3 char's embeddings.
# so, each trigram's embedding is of shape (1, 3, 2).
# and, we have 32 trigrams in a single batch, i.e. every 32 tri-grams, our network updates it's parameters.
# so, when we concatenate 32 trigrams, we get a tensor of shape (32, 3, 2).

# this is all well and good, but, this embeddings of (32, 3, 2) can't be fed into the hidden layer that's supposed to come next. Because it's a 3-dim tensor, and by convention, neural net layers expect a 2-d tensor.
# so, we flatten (32, 3, 2) into (32, 6), and then we can feed it into the hidden layer.
# note: i'm saying 32 in (32, ...) shaped tensors here, because 32 is going to be my batch size, but realistically, it could be any size.

emb_flat = emb.view(-1, 6)
emb_flat.shape

In [ ]:
# now, we pass in the flattened tensor through the weight matrix of the hidden layer.
# how are we going to initialize it tho? what dims should it be?
# let's see. first, we have the input embeddings tensor, which is going to be of shape (32, 6), so, the hidden layer should be of dim (6, ..)
# the second dimension, i.e. the number of neurons, is up to us. It can be 100, or 300, or 1000, etc. let's stick to 100, for now.

activ = torch.tanh(emb_flat @ W1 + b1)
activ.shape

In [ ]:
# now that we have the activations of the hidden layer, we want to now, pass it through the last layer, i.e. the linear layer.
# since the activations are of the shape (32, 100), the linear layer is going to have to be of the shape (100, 27), becuase, one: we match end dims.
# and two, we want our output to basically be probabilities of all characters. So, we want a (32, 27) shaped output, which is nothing but logits.
# out of the (32, 27), i.e 32 rows, each row corresponds to each input. (remember, we had 32 tri-grams as inputs.)

logits = activ @ W2 + b2
logits.shape


In [ ]:
# now, to calculate the loss.
# we want to calculate either of the -ve log likelihood loss or the cross entropy loss.
# so, first, we have to calculate the probs, which is done using softmax.

exps = torch.exp(logits)
probs = exps / torch.sum(exps, dim=1, keepdim=True)

In [ ]:
loss = F.cross_entropy(logits, Y)
loss

In [ ]:
# now, for the backward pass
for p in params:
    p.grad = None

loss.backward() # fills gradients

In [ ]:
# now, time to update the weights and biases using the gradients that we just calculated.
# we have to update W2, b2, W1, b1 and C
# all of these are our parameters

for p in params:
    p.data += -0.1 * p.grad

In [7]:
# now, for the training loop.

for i in range(100):
    emb = C[X]
    emb_flat = emb.view(-1, 6)
    activ = torch.tanh(emb_flat @ W1 + b1) # hidden layer
    logits = activ @ W2 + b2 # linear layer

    loss = F.cross_entropy(logits, Y) # loss
    print(i, loss.item())

    for p in params:
        p.grad = None

    loss.backward() # fills gradients

    # update
    for p in params:
        p.data += -0.1 * p.grad

0 16.140092849731445
1 14.957382202148438
2 14.020736694335938
3 13.277274131774902
4 12.622754096984863
5 12.013656616210938
6 11.451870918273926
7 10.949630737304688
8 10.505123138427734
9 10.11467170715332
10 9.773237228393555
11 9.4664888381958
12 9.18223762512207
13 8.915148735046387
14 8.66329574584961
15 8.425790786743164
16 8.20168685913086
17 7.989971160888672
18 7.789975643157959
19 7.601363182067871
20 7.423891067504883
21 7.257176399230957
22 7.1006293296813965
23 6.953512668609619
24 6.81503438949585
25 6.6844000816345215
26 6.560871124267578
27 6.44378137588501
28 6.332564353942871
29 6.226753234863281
30 6.125980377197266
31 6.029956817626953
32 5.938458442687988
33 5.851302623748779
34 5.768326759338379
35 5.689382076263428
36 5.614318370819092
37 5.542975425720215
38 5.475181579589844
39 5.410749912261963
40 5.3494768142700195
41 5.291150093078613
42 5.235551357269287
43 5.182466983795166
44 5.131688117980957
45 5.0830230712890625
46 5.036292552947998
47 4.991336822509

In [ ]:
# ok, now, we have to implement mini-batching. How do we do that?